In [0]:
# 1. The "Ingestion & Processing" Layer (PySpark)
# In Databricks, Spark is native. This code simulates receiving "dirty" sensor data from a stream (Kafka) and cleaning it for storage in a Data Lake (Delta Lake/S3).

# --- STEP 1: Spark Data Cleaning (The 'Compute' Phase) ---
from pyspark.sql.functions import col, current_timestamp, avg

# Simulating incoming sensor data (Pressure from Water Network)
data = [("Sensor_A1", 45.2), ("Sensor_A2", 38.5), ("Sensor_A1", 44.8), ("Sensor_B1", 12.0)]
columns = ["sensor_id", "pressure_psi"]

df = spark.createDataFrame(data, columns)
display(df)

In [0]:
# Logic: Identify low pressure (Potential Leak < 40 PSI)
cleaned_df = df.withColumn("ingestion_time", current_timestamp()) \
               .filter(col("pressure_psi") > 0) # Remove noise

# Aggregating to find average pressure by sector
stats_df = cleaned_df.groupBy("sensor_id").agg(avg("pressure_psi").alias("avg_pressure"))

stats_df.show()
# This results would typically be saved to Snowflake or a Delta Table here.

In [0]:
# 2. The "Advanced Intelligence" Layer (Ray)
# While Spark handles the table above, we use Ray to run a parallel simulation or a complex model that doesn't fit into a standard SQL-like table. Note: In Databricks, you can install Ray via %pip install ray.

# --- STEP 2: Ray Distributed Task (The 'Analyze' Phase) ---
!pip install ray
import ray

In [0]:
# Initialize Ray (In Databricks, this connects to the cluster nodes)
if not ray.is_initialized():
    ray.init()

@ray.remote
def predict_leak_risk(sensor_id, avg_pressure):
    """
    Complex logic that might use a pre-trained TensorFlow 
    or Scikit-learn model to predict the probability of a burst.
    """
    import time
    time.sleep(1) # Simulating heavy computation
    risk_score = 1.0 if avg_pressure < 35 else 0.1
    return {sensor_id: risk_score}

# Execute the function in parallel for all sensors found by Spark
sensor_list = stats_df.collect()
futures = [predict_leak_risk.remote(row['sensor_id'], row['avg_pressure']) for row in sensor_list]

# Get results back
final_risks = ray.get(futures)
print(f"Real-time Risk Assessment: {final_risks}")

In [0]:
# Handling the GCS connection carefully for Community Edition
import pandas as pd

if ray.is_initialized():
    ray.shutdown()

# Initialize Ray locally on the driver node
ray.init(ignore_reinit_error=True)

@ray.remote
def analyze_leak_risk(sensor_id, pressure):
    """
    Complex Distributed Task: 
    In a real scenario, this would load a Scikit-learn or TensorFlow model.
    """
    import time
    time.sleep(0.5) # Simulate complex model inference
    
    # Logic: Risk increases as pressure drops below 35 PSI
    risk_level = "HIGH" if pressure < 35 else "LOW"
    return {"id": sensor_id, "pressure": round(pressure, 2), "risk": risk_level}

# Convert Spark DataFrame to pandas for iteration
results_df = stats_df.toPandas()

# Launching Ray Tasks in Parallel
print("\n--- Launching Ray Distributed Tasks ---")
futures = [analyze_leak_risk.remote(row['sensor_id'], row['avg_pressure']) for _, row in results_df.iterrows()]

# Collect results
results = ray.get(futures)

# --- FINAL OUTPUT ---
results_df = pd.DataFrame(results)
print("\n--- Final Risk Intelligence Table ---")
print(results_df)

# Shutdown Ray to free up memory for the next run
ray.shutdown()

In [0]:
from pyspark.sql.functions import udf, col, avg
from pyspark.sql.types import StringType
import time

# 2. THE CUSTOM LOGIC (The "Brain")
# This is where your AI model or complex math would live
def analyze_risk_logic(pressure):
    """
    Simulates a complex Python model.
    Spark will run this in parallel on all worker nodes.
    """
    if pressure is None: return "UNKNOWN"
    # Imagine calling a scikit-learn .predict() here
    if pressure < 30:
        return "CRITICAL LEAK"
    elif pressure < 40:
        return "WARNING"
    else:
        return "STABLE"

# Register the Python function as a Spark UDF
risk_udf = udf(analyze_risk_logic, StringType())

# 3. THE EXECUTION
# We aggregate the data and THEN apply the intelligence
final_results = df.groupBy("sensor_id") \
                  .agg(avg("pressure_psi").alias("avg_p")) \
                  .withColumn("risk_status", risk_udf(col("avg_p")))

# Display results
print("--- Integrated Spark Risk Intelligence ---")
final_results.show()

# 4. EXPORT TO SQL (For Dashboards)
final_results.createOrReplaceTempView("v_leak_intelligence")

# Bank Example
## Credit Risk

In [0]:
# 1. Ingestion: Upload the credit_risk_dataset.csv to a Unity Catalog Volume.

# * Bronze (Raw): Read the CSV exactly as it is. This is your "Source of Truth."
# * Silver (Cleaned): This is where you handle the specific issues in this Kaggle set (imputing missing interest rates, removing outliers in person age, etc.).
# * Gold (Risk Features): Creating the final "Default Probability" flags for the business.

# 2. Implementation Code (Spark-Native)

# First Step:  
## GO to Create Medallion (schemas created)
$$bronze \rightarrow \rightarrow silver \rightarrow gold$$

In [0]:
# Step A: Bronze (The Raw Table)
# We read from the Volume and save it as a permanent table in the Bronze Schema.
# Path to your uploaded Kaggle file in the Volume
csv_path = "/Volumes/workspace/bronze/landing_zone/credit_risk_dataset.csv"

# Read Raw
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_path)

# Save to Bronze Schema
df_raw.write.mode("overwrite").saveAsTable("workspace.bronze.credit_risk_raw")

In [0]:
display(df_raw)

In [0]:
%sql
select person_age 
from workspace.bronze.credit_risk_raw;

select * from workspace.bronze.credit_risk_raw
where person_age > 50

In [0]:
# Step B: Silver (The Cleaned Table)
# We apply the "Bank Rules" (handling nulls and filtering outliers) and save it to the Silver Schema.

from pyspark.sql.functions import when, col, mean, split

# Read from Bronze

bronze_df = spark.read.table("workspace.bronze.credit_risk_raw")

# Cleaning Logic: Fill nulls & remove age outliers
avg_rate = bronze_df.select(mean("loan_int_rate")).collect()[0][0]

df_silver = bronze_df.withColumn("loan_int_rate", 
    when(col("loan_int_rate").isNull(), avg_rate).otherwise(col("loan_int_rate"))) \
    .filter(col("person_age") < 100)

# Save to Silver
df_silver.write.mode("overwrite").saveAsTable("workspace.silver.credit_risk_cleaned")

In [0]:
# Layer 3: Gold (The "Knowledge")
# We apply the high-level business logic (Credit Risk segments) and save to gold.
silver_df = spark.read.table("workspace.silver.credit_risk_cleaned")

# Business Logic: Segmenting users by risk
df_gold = silver_df.withColumn("risk_segment", 
    when((col("loan_percent_income") > 0.3) & (col("person_home_ownership") == 'RENT'), "HIGH_RISK")
    .otherwise("STANDARD_RISK")
)

# Save to Gold
df_gold.write.mode("overwrite").saveAsTable("workspace.gold.credit_risk_final")